# NBA 3.1 - 07 Physician attributes features

**Assessment:** `NBA 3.0 Static Feature Analysis/07_2_physician_attributes_data_assessment.md` | **Universe:** `HCP_Universe_without_call_hcps` (294,727) | **Cohorts:** 28 claims cohorts c1-c28 | **Writers (signal only):** KER-10 `active_nbrx_writers` in `kerendia_metric_views.mv_kerendia_nbrx` | **Output:** `kerendia_nba_3_1_volume/.../notebook_output/hcp_features/physician_attributes_features`

_Summary is filled in after the run (see the gate cell for the live results)._

## Decisions taken in this notebook

The feature list follows `07_2_physician_attributes_data_assessment.md` section 4 (Build and Test rows) and section 6 (recommended order). The layout follows `33_colocation_idn_features` (same universe, cohorts, keys and output folder). The feature logic is taken from the 3.1 draft `07_phy_static_features_creation` (Feature Creation Static folder), which 07_2 checked.

| # | Topic | Decision | Why |
| --- | --- | --- | --- |
| N1 | Sources | `MDM_STG_HCP_PRFL` and `MDM_STG_HCP_ACCT_AFFL` are read from **Snowflake** (as in the draft). BEST_AFFL and the account profile are read from UC | ACCT_AFFL has no UC copy (07_2 sources, DR-06). The UC copy of PRFL (`pace_ker_in_house_schema.mdm_stg_hcp_prfl`) is a different snapshot: against the draft's Snowflake read it differs for 830 gender, 54 degree, 82 secondary specialty and 28 grad year values among 294,493 universe HCPs |
| N2 | Build set | HCP_DEGREE_NP_PA, HCP_GENDER_F, HCP_SECONDARY_SPEC_CARDIORENAL, HCP_EXPERIENCE, AFFL_NEPH_ACCT_COUNT, AFFL_CARD_ACCT_COUNT, AFFL_DISTINCT_ACCT_COUNT, DIALYSIS_CENTER_FLAG | 07_2 section 4, Build rows |
| N3 | Test set | AFFL_ENDO_ACCT_COUNT, TEACHING_HOSP_FLAG, LONG_TERM_CARE_FACIL_FLAG, three description flags (HEALTH_SYSTEM, ADMIN_ONLY, ACOS), four PRFL code flags (SELF_EMPLOYED_SOLO_PRACTICE, MPA_INACTIVE, PRACTICE_TYPE_RESIDENT, PRACTICE_TYPE_NOT_CLASSIFIED), BEST_AFFL EMPLOYMENT_FLAG one-hot (Y / N / NA), ACCT_MD_DO_COUNT | 07_2 section 4, Test rows |
| N4 | Not built | HCP_AGE, AFFL_COUNT, AFFL_HOSPITAL_COUNT, AFFL_IDN_ACO_FLAG, AFFL_POST_ACUTE_FLAG, PRIMARY_*, the other 10 description flags and the other practice / MPA / employment one-hots | 07_2 section 4, Skip rows |
| N5 | IDN features | IDN_PEER_KERENDIA_SHARE (+ zero flag) and HCP_IN_IDN are **not** built here | Built in `33_colocation_idn_features` as `COLOC_IDN_PEER_SHARE_3M`, `COLOC_IDN_PEER_SHARE_ZERO_FLAG` and `COLOC_HCP_IN_IDN` (07_2 ownership note; colocation D4) |
| N6 | Experience anchor | `HCP_EXPERIENCE` = year of the cohort `prediction_month` - GRAD_YEAR; grad years after that year (or <= 1900) are NULL. This is the only feature that changes between cohorts | Fixes the hard-coded 2026 in the draft (07_2 section 1.1 row 3); no future grad year leaks into an early cohort |
| N7 | Code flags | Fixed MDM codes: TYPE_OF_PRACTICE 20 = RESIDENT, 99 = NOT_CLASSIFIED; MPA 90 = INACTIVE; PLACE_OF_EMPLOYMENT 2 = SELF_EMPLOYED_SOLO_PRACTICE. Checked against the MDM reference table below. 1/0 for HCPs with a PRFL row, NULL without one | Same codes as the draft's decoded columns; a fixed schema does not depend on the 1% prevalence rule |
| N8 | Missing states | Affiliation counts and flags NULL for HCPs with no ACCT_AFFL row; description flags 0 for HCPs with affiliations but not that type. EMPLOYMENT_FLAG one-hots NULL without a BEST_AFFL rank-1 row | 07_2 section 1.3: no-affiliation HCPs write at 5.9%, so NULL is informative |
| N9 | Account size | `ACCT_MD_DO_COUNT` = `COUNT_MD_DO` of the BEST_AFFL rank-1 account in `mdm_stg_acct_prfl_ker` when > 0, else NULL | 07_2 section 3.3 tested `COUNT_MD_DO > 0`; DR-07 notes 0 means not filled |
| N10 | Grain, keys, output | One row per `PROVIDER_ID` x `cohort_id`; `SVC_DT_M` = cohort `lookback_end_date`; `HCP_COHORT_ID = PROVIDER_ID_cohort_id`; written to `.../notebook_output/hcp_features/physician_attributes_features` | Same as colocation N3 / N4. The 3.1 draft output (`static_features/07_phy_static_features`) is not touched |
| N11 | Snapshot | ACCT_AFFL, BEST_AFFL and the account profile are current snapshots applied to all cohorts | Colocation plan D7; history requested in DR-01 / DR-06 |
| N12 | Signal restatement | KER-10 writers (D8): cohort c17, look-back 2025-07 to 2025-09, outcome = KER-10 writer in 2025-10 to 2026-09; new-writer view = not counted by KER-10 in the look-back | Same window as colocation N13 and `08_census_features` |

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
import json, re, builtins
# 00_connection star-imports pyspark functions, which shadows these Python built-ins; restore them
abs, max, min, round, sum = builtins.abs, builtins.max, builtins.min, builtins.round, builtins.sum
import numpy as np
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql.window import Window

CAT = "`ph-com-ai-us-prod-catalog-2553213839599380`"
VOL31 = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume"

HCP_UNIVERSE_PATH = f"{VOL31}/HCP_Universe_without_call_hcps"        # 294,727 HCPs
COHORTS_DIR = f"{VOL31}/cohorts/claims"                              # 28 claims cohorts (N10)
MV_NBRX = f"{CAT}.`kerendia_metric_views`.`mv_kerendia_nbrx`"        # KER-10 writers, signal check only (N12)
MAPPING = f"{CAT}.kerendia_nba_3.source_provider_bhid_npi_mapping"
BEST_AFFL = f"{CAT}.pace_ker_in_house_schema.cust_mdm_best_affl_publish_pd_outp_static"
ACCT_PRFL = f"{CAT}.pace_ker_in_house_schema.mdm_stg_acct_prfl_ker"
DRAFT_OUTPUT_PATH = f"{VOL31}/hcp_level_model/dev_pipeline/notebook_output/static_features/07_phy_static_features"  # read only, for comparison

OUT_DIR = f"{VOL31}/hcp_level_model/dev_pipeline/notebook_output/hcp_features"
OUTPUT_PATH = f"{OUT_DIR}/physician_attributes_features"             # N10
assert "kerendia_nba_3_1_volume" in OUTPUT_PATH and OUTPUT_PATH != DRAFT_OUTPUT_PATH, "never write outside the 3.1 volume or over the draft"

# N7: fixed MDM codes behind the kept code flags
CODE_FLAGS = {"HCP_PRACTICE_TYPE_RESIDENT": ("TYPE_OF_PRACTICE", "20", "HCPPracticeType"),
              "HCP_PRACTICE_TYPE_NOT_CLASSIFIED": ("TYPE_OF_PRACTICE", "99", "HCPPracticeType"),
              "HCP_MPA_INACTIVE": ("MPA", "90", "HCPMajorProfActivity"),
              "HCP_EMPLOYMENT_SELF_EMPLOYED_SOLO_PRACTICE": ("PLACE_OF_EMPLOYMENT", "2", "HCPPlaceOfEmployment")}
CARDIORENAL_SPEC2 = ["CD", "IC", "AHF", "HFT", "NEP", "END", "DIA"]
DESC_FLAGS = ["HCP_AFFL_DESCRIPTION_ORGANIZATION_HEALTH_SYSTEM", "HCP_AFFL_DESCRIPTION_ORGANIZATION_ADMIN_ONLY",
              "HCP_AFFL_DESCRIPTION_ORGANIZATION_ACCOUNTABLE_CARE_ORGANIZATIONS_ACOS"]
EMP_FLAGS = {"BEST_EMPLOYMENT_FLAG_Y": "Y", "BEST_EMPLOYMENT_FLAG_N": "N", "BEST_EMPLOYMENT_FLAG_NA": "NA"}

BUILD_COLS = ["HCP_DEGREE_NP_PA", "HCP_GENDER_F", "HCP_SECONDARY_SPEC_CARDIORENAL", "HCP_EXPERIENCE",
              "AFFL_NEPH_ACCT_COUNT", "AFFL_CARD_ACCT_COUNT", "AFFL_DISTINCT_ACCT_COUNT", "DIALYSIS_CENTER_FLAG"]
TEST_COLS = (["AFFL_ENDO_ACCT_COUNT", "TEACHING_HOSP_FLAG", "LONG_TERM_CARE_FACIL_FLAG"] + DESC_FLAGS + list(CODE_FLAGS)
             + list(EMP_FLAGS) + ["ACCT_MD_DO_COUNT"])
FEATURE_COLS = BUILD_COLS + TEST_COLS
STATIC_COLS = [c for c in FEATURE_COLS if c != "HCP_EXPERIENCE"]    # N6: only experience varies by cohort
assert len(BUILD_COLS) == 8 and len(TEST_COLS) == 14
KEYS = ["PROVIDER_ID", "cohort_id", "SVC_DT_M", "HCP_COHORT_ID"]

CHECKS = {}
def record(check_id, passed, detail):
    CHECKS[check_id] = {"passed": bool(passed), "detail": detail}
    print(f"[{'PASS' if passed else 'FAIL'}] {check_id}: {detail}")

def clean_desc(s):
    """Same name cleaning as the draft (HCP_AFFL_DESCRIPTION_<cleaned description>)."""
    return re.sub(r"_+", "_", re.sub(r"[^A-Z0-9]", "_", str(s).upper().replace("/", " OR ").replace("+", " AND "))).strip("_")

print(f"{len(FEATURE_COLS)} feature columns ({len(BUILD_COLS)} Build + {len(TEST_COLS)} Test) -> {OUTPUT_PATH}")

In [0]:
cohorts = pd.read_parquet(f"{COHORTS_DIR}/cohorts.parquet")
cohorts = cohorts.sort_values("cohort_id", key=lambda s: s.str[1:].astype(int)).reset_index(drop=True)
with open(f"{COHORTS_DIR}/metadata.json") as f:
    meta = json.load(f)
assert meta["feature_family"] == "claims" and len(cohorts) == meta["cohort_count"] == 28
for c in ["prediction_month", "lookback_end_date"]:
    cohorts[c] = pd.to_datetime(cohorts[c]).dt.date
cohorts["anchor_year"] = [d.year for d in cohorts.prediction_month]          # N6
coh = spark.createDataFrame(cohorts[["cohort_id", "dataset", "prediction_month", "lookback_end_date", "anchor_year"]])

universe = spark.read.parquet(HCP_UNIVERSE_PATH).select(F.col("PROVIDER_ID").cast("string")).distinct()
n_hcp = universe.count()
assert n_hcp == 294727, n_hcp
print(f"HCP universe {n_hcp:,} | {len(cohorts)} claims cohorts {cohorts.cohort_id.iloc[0]}..{cohorts.cohort_id.iloc[-1]} | anchor years {sorted(cohorts.anchor_year.unique())}")

## Phase 0: Sources and coverage checks

Snowflake reads (N1), MDM code check (N7) and coverage against `07_2` before any feature is built. Results go to `CHECKS`.

In [0]:
mp = (spark.table(MAPPING).select(F.col("PROVIDER_ID").cast("string"), F.col("BH_ID").cast("string"), F.col("NPI_NUMBER").cast("string"))
      .filter("BH_ID IS NOT NULL").dropDuplicates(["PROVIDER_ID"]))
spine = universe.join(mp, "PROVIDER_ID", "left")

# PRFL: one row per NPI, most recent by CREATED_DATE (draft query)
prfl_raw = Get_Data_Snowflakes("""
  WITH cte AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY NPI_NUM ORDER BY CREATED_DATE DESC) AS rn
    FROM PHCDW.PHCDW_STG.MDM_STG_HCP_PRFL WHERE NPI_NUM IS NOT NULL)
  SELECT NPI_NUM AS NPI_NUMBER, MEDICAL_DEGREE_1, GENDER, SPECIALTY_2, GRAD_YEAR, TYPE_OF_PRACTICE, MPA, PLACE_OF_EMPLOYMENT
  FROM cte WHERE rn = 1""")
# ACCT_AFFL: all HCP-account affiliations (Snowflake only, DR-06)
affl_raw = Get_Data_Snowflakes("""
  SELECT BH_ID, BHO_ID, ACCT_NAME, MAJOR_CLASS_OF_TRADE, DESCRIPTION, IS_PRIMARY_RELATIONSHIP__V, SPECIALTY
  FROM PHCDW.PHCDW_STG.MDM_STG_HCP_ACCT_AFFL""")
# N7: MDM reference decode of the fixed codes
ref = Get_Data_Snowflakes("""
  SELECT REFERENCE_TYPE, REFERENCE_CD, REFERENCE_DESC FROM PHCDW.PUB.MDM_OUTB_MDM_REFERENCE
  WHERE REFERENCE_TYPE IN ('HCPPracticeType', 'HCPMajorProfActivity', 'HCPPlaceOfEmployment')""").toPandas()
decode = {(r.REFERENCE_TYPE, str(r.REFERENCE_CD).strip()): r.REFERENCE_DESC for r in ref.itertuples()}
bad_codes = {name: decode.get((rt, code)) for name, (_, code, rt) in CODE_FLAGS.items()
             if not name.endswith("_" + clean_desc(decode.get((rt, code), "")))}
record("S1 MDM codes match flag names (N7)", not bad_codes, f"codes checked: {len(CODE_FLAGS)}, mismatches: {bad_codes if bad_codes else 'none'}")

prfl = spine.join(prfl_raw.withColumn("NPI_NUMBER", F.col("NPI_NUMBER").cast("string")), "NPI_NUMBER", "inner").dropDuplicates(["PROVIDER_ID"])
affl = spine.select("PROVIDER_ID", "BH_ID").join(affl_raw.withColumn("BH_ID", F.col("BH_ID").cast("string")), "BH_ID", "inner")
best = (spark.table(BEST_AFFL).filter("BRAND_RANK = 1")
        .select(F.col("BH_ID").cast("string"), F.col("BHO_ID").cast("string"), F.trim(F.col("EMPLOYMENT_FLAG")).alias("EMPLOYMENT_FLAG")).distinct())
multi_best = best.groupBy("BH_ID").count().filter("count > 1").count()
record("S1 one rank-1 BEST_AFFL row per BH_ID", multi_best == 0, f"BH_IDs with more than one rank-1 row: {multi_best}")

cov = {"with_bh": spine.filter("BH_ID IS NOT NULL").count(), "with_prfl": prfl.count(),
       "with_affl": affl.select("PROVIDER_ID").distinct().count(),
       "with_best": spine.join(best, "BH_ID", "inner").select("PROVIDER_ID").distinct().count()}
print(cov)
# 07_2: 283,022 HCPs with at least one affiliation (11,705 NULL); 282,881 with a BEST_AFFL rank-1 row
# ACCT_AFFL is a live Snowflake table, so it may refresh after 07_2: accept up to 0.5%; BEST_AFFL (UC) must match exactly
record("S1 coverage = 07_2", abs(cov["with_affl"] - 283022) / 283022 <= 0.005 and cov["with_best"] == 282881,
       f"affiliations {cov['with_affl']:,} (07_2: 283,022), BEST_AFFL {cov['with_best']:,} (282,881), PRFL {cov['with_prfl']:,}, BH_ID {cov['with_bh']:,}")

## Phase 1: Build features

Profile flags, code flags (N7), affiliation counts and flags (N8), BEST_AFFL employment one-hot and account size (N9). Built once per HCP; `HCP_EXPERIENCE` is added per cohort when saved (N6).

In [0]:
deg = F.upper(F.trim(F.col("MEDICAL_DEGREE_1")))
sp2 = F.upper(F.trim(F.col("SPECIALTY_2")))
grad = F.col("GRAD_YEAR").cast("int")
prfl_f = prfl.select(
    "PROVIDER_ID",
    F.when(deg.isNull(), None).when(deg.isin("NP", "PA", "APRN"), 1).otherwise(0).alias("HCP_DEGREE_NP_PA"),
    F.when(F.col("GENDER") == "F", 1).when(F.col("GENDER") == "M", 0).alias("HCP_GENDER_F"),     # U / missing -> NULL
    F.when(sp2.isin(*CARDIORENAL_SPEC2), 1).otherwise(0).alias("HCP_SECONDARY_SPEC_CARDIORENAL"),
    F.when(grad > 1900, grad).alias("_GRAD_YEAR"),                                                # N6: anchored per cohort later
    *[F.when(F.trim(F.col(src)) == code, 1).otherwise(0).alias(name) for name, (src, code, _) in CODE_FLAGS.items()])

MCT, DESC = F.col("MAJOR_CLASS_OF_TRADE"), F.col("DESCRIPTION")
affl_f = affl.groupBy("PROVIDER_ID").agg(
    F.countDistinct("BHO_ID").alias("AFFL_DISTINCT_ACCT_COUNT"),
    F.countDistinct(F.when(F.col("SPECIALTY") == "NEP", F.col("BHO_ID"))).alias("AFFL_NEPH_ACCT_COUNT"),
    F.countDistinct(F.when(F.col("SPECIALTY") == "CD", F.col("BHO_ID"))).alias("AFFL_CARD_ACCT_COUNT"),
    F.countDistinct(F.when(F.col("SPECIALTY") == "END", F.col("BHO_ID"))).alias("AFFL_ENDO_ACCT_COUNT"),
    F.max(F.when(MCT == "Dialysis & Infusion Centers", 1).otherwise(0)).alias("DIALYSIS_CENTER_FLAG"),
    F.max(F.when(DESC == "Organization, CMS Teaching Hospital", 1).otherwise(0)).alias("TEACHING_HOSP_FLAG"),
    F.max(F.when(DESC == "Organization, Nursing Home / Long Term Care", 1).otherwise(0)).alias("LONG_TERM_CARE_FACIL_FLAG"))
# Description flags: same cleaned names as the draft pivot; 0 for HCPs with affiliations but not that type (N8)
desc_map = {r["DESCRIPTION"]: "HCP_AFFL_DESCRIPTION_" + clean_desc(r["DESCRIPTION"])
            for r in affl.select("DESCRIPTION").filter(DESC.isNotNull()).distinct().collect()}
desc_keep = {d: n for d, n in desc_map.items() if n in DESC_FLAGS}
assert sorted(desc_keep.values()) == sorted(DESC_FLAGS), f"description flags not found: {set(DESC_FLAGS) - set(desc_keep.values())}"
desc_f = affl.groupBy("PROVIDER_ID").agg(*[F.max(F.when(DESC == d, 1).otherwise(0)).alias(n) for d, n in desc_keep.items()])
print("Description flags from:", desc_keep)

# BEST_AFFL rank-1: EMPLOYMENT_FLAG one-hot and account size (N9)
acct = spark.table(ACCT_PRFL).select(F.col("BHO_ID").cast("string"), F.col("COUNT_MD_DO").cast("double").alias("_MD_DO")).dropDuplicates(["BHO_ID"])
best_f = (spine.select("PROVIDER_ID", "BH_ID").join(best, "BH_ID", "inner").join(acct, "BHO_ID", "left")
          .select("PROVIDER_ID", *[F.when(F.col("EMPLOYMENT_FLAG") == v, 1).otherwise(0).alias(n) for n, v in EMP_FLAGS.items()],
                  F.when(F.col("_MD_DO") > 0, F.col("_MD_DO")).alias("ACCT_MD_DO_COUNT")))

feat_hcp = (universe.join(prfl_f, "PROVIDER_ID", "left").join(affl_f, "PROVIDER_ID", "left")
            .join(desc_f, "PROVIDER_ID", "left").join(best_f, "PROVIDER_ID", "left"))
feat_hcp = feat_hcp.select("PROVIDER_ID", "_GRAD_YEAR", *STATIC_COLS)
feat_pd = feat_hcp.toPandas()
print(f"{len(feat_pd):,} HCP rows, {feat_pd.PROVIDER_ID.nunique():,} distinct HCPs, {len(STATIC_COLS)} static features + HCP_EXPERIENCE per cohort")

In [0]:
fp = feat_pd
record("S2 one row per HCP", len(fp) == n_hcp and fp.PROVIDER_ID.is_unique, f"{len(fp):,} rows, unique PROVIDER_ID: {fp.PROVIDER_ID.is_unique}")

# Counts reported in 07_2 sections 3.1-3.3 (computed on the draft output and the UC account tables)
counts = {"HCP_DEGREE_NP_PA = 1": (int((fp.HCP_DEGREE_NP_PA == 1).sum()), 109809),
          "HCP_GENDER_F = 1": (int((fp.HCP_GENDER_F == 1).sum()), 154244),
          "HCP_SECONDARY_SPEC_CARDIORENAL = 1": (int((fp.HCP_SECONDARY_SPEC_CARDIORENAL == 1).sum()), 12697),
          "AFFL_NEPH_ACCT_COUNT > 0": (int((fp.AFFL_NEPH_ACCT_COUNT > 0).sum()), 15619),
          "AFFL_CARD_ACCT_COUNT > 0": (int((fp.AFFL_CARD_ACCT_COUNT > 0).sum()), 47406),
          "AFFL_ENDO_ACCT_COUNT > 0": (int((fp.AFFL_ENDO_ACCT_COUNT > 0).sum()), 10279),
          "DIALYSIS_CENTER_FLAG = 1": (int((fp.DIALYSIS_CENTER_FLAG == 1).sum()), 6919),
          "TEACHING_HOSP_FLAG = 1": (int((fp.TEACHING_HOSP_FLAG == 1).sum()), 116568),
          "LONG_TERM_CARE_FACIL_FLAG = 1": (int((fp.LONG_TERM_CARE_FACIL_FLAG == 1).sum()), 9686),
          "AFFL_DISTINCT_ACCT_COUNT filled": (int(fp.AFFL_DISTINCT_ACCT_COUNT.notna().sum()), 283022),
          "BEST_EMPLOYMENT_FLAG_Y = 1": (int((fp.BEST_EMPLOYMENT_FLAG_Y == 1).sum()), 230078),
          "BEST_EMPLOYMENT_FLAG_N = 1": (int((fp.BEST_EMPLOYMENT_FLAG_N == 1).sum()), 37220),
          "BEST_EMPLOYMENT_FLAG_NA = 1": (int((fp.BEST_EMPLOYMENT_FLAG_NA == 1).sum()), 15583),
          "ACCT_MD_DO_COUNT filled": (int(fp.ACCT_MD_DO_COUNT.notna().sum()), 196044)}
cnt = pd.DataFrame([{"measure": k, "this_notebook": v[0], "07_2": v[1], "diff": v[0] - v[1]} for k, v in counts.items()])
max_rel = float((cnt["diff"].abs() / cnt["07_2"]).max())
# Snowflake tables can refresh after 07_2 (2026-11-18), so a difference up to 0.5% is accepted and shown
record("S2 counts = 07_2 (within 0.5%)", max_rel <= 0.005, f"{len(cnt)} counts, exact matches {int((cnt['diff'] == 0).sum())}, largest relative difference {max_rel:.3%}")
display(cnt)

# Same logic as the draft: compare every shared column for the universe HCPs (draft = Snowflake read of 2026-11-13)
draft = spark.read.format("delta").load(DRAFT_OUTPUT_PATH).toPandas()
shared = [c for c in STATIC_COLS if c in draft.columns]
m = fp.merge(draft[["PROVIDER_ID"] + shared + ["HCP_EXPERIENCE"]].rename(columns={"HCP_EXPERIENCE": "HCP_EXPERIENCE__draft"}),
             on="PROVIDER_ID", how="inner", suffixes=("", "__draft"))
diff_rows = {c: int((~((m[c] == m[f"{c}__draft"]) | (m[c].isna() & m[f"{c}__draft"].isna()))).sum()) for c in shared}
exp26 = np.where((m._GRAD_YEAR <= 2026), 2026 - m._GRAD_YEAR, np.nan)
diff_rows["HCP_EXPERIENCE (anchor 2026)"] = int((~((exp26 == m.HCP_EXPERIENCE__draft) | (np.isnan(exp26) & m.HCP_EXPERIENCE__draft.isna()))).sum())
share_diff = max(diff_rows.values()) / len(m)
record("S2 same values as the 3.1 draft (within 0.5% of HCPs)", share_diff <= 0.005,
       f"{len(m):,} HCPs compared on {len(diff_rows)} columns; rows that differ per column: {diff_rows}")
print("Not compared (new in this notebook):", [c for c in STATIC_COLS if c not in shared])

# Missing states (N8): no-affiliation HCPs have NULL affiliation features; description flags 0/1 when affiliated
affl_cols = ["AFFL_DISTINCT_ACCT_COUNT", "AFFL_NEPH_ACCT_COUNT", "AFFL_CARD_ACCT_COUNT", "AFFL_ENDO_ACCT_COUNT",
             "DIALYSIS_CENTER_FLAG", "TEACHING_HOSP_FLAG", "LONG_TERM_CARE_FACIL_FLAG"] + DESC_FLAGS
no_affl = fp[fp.AFFL_DISTINCT_ACCT_COUNT.isna()]
v1 = int(no_affl[affl_cols].notna().any(axis=1).sum())
v2 = int(fp.loc[fp.AFFL_DISTINCT_ACCT_COUNT.notna(), DESC_FLAGS].isna().any(axis=1).sum())
v3 = int((fp.AFFL_NEPH_ACCT_COUNT > fp.AFFL_DISTINCT_ACCT_COUNT).sum() + (fp.AFFL_CARD_ACCT_COUNT > fp.AFFL_DISTINCT_ACCT_COUNT).sum())
record("S2 missing-state rules (N8)", v1 == 0 and v2 == 0 and v3 == 0,
       f"{len(no_affl):,} HCPs without affiliation; violations: no-affl filled {v1}, affiliated with NULL type flag {v2}, specialty count > distinct {v3}")

## Phase 2: Signal restated on KER-10 writers (N12)

Same window as colocation and census: cohort c17, look-back 2025-07-01 to 2025-09-30, outcome = counted by KER-10 in any month 2025-10 to 2026-09. Flags and counts: writer rate with vs without (counts as > 0 vs 0, as in 07_2). `HCP_EXPERIENCE` (anchored to 2025 for c17), `AFFL_DISTINCT_ACCT_COUNT` and `ACCT_MD_DO_COUNT`: Q1 vs Q5 and Spearman rho. The check is that the 07_2 **direction** holds where 07_2 showed a clear full-universe gap.

In [0]:
SIG_COHORT = "c17"
row = cohorts[cohorts.cohort_id == SIG_COHORT].iloc[0]
lb_start = (pd.Timestamp(row.lookback_end_date) - pd.DateOffset(months=2)).date()
lb_end = (pd.Timestamp(row.lookback_end_date) + pd.offsets.MonthEnd(0)).date()
out_start = (pd.Timestamp(row.lookback_end_date) + pd.DateOffset(months=1)).date()
out_end = (pd.Timestamp(out_start) + pd.DateOffset(months=11)).date()

mv_rows = spark.sql(f"""
    SELECT CAST(`provider_id` AS STRING) AS PROVIDER_ID, `week_ending`, `month`, MEASURE(`active_nbrx_writers`) AS W
    FROM {MV_NBRX}
    WHERE `week_ending` >= DATE'{lb_start}'
    GROUP BY ALL
    HAVING W > 0""")
mv_max = mv_rows.agg(F.max("month")).collect()[0][0]
assert mv_max >= out_end, f"metric view ends {mv_max}, outcome needs {out_end}"
own_lb = (mv_rows.filter((F.col("week_ending") >= F.lit(lb_start)) & (F.col("week_ending") <= F.lit(lb_end)))
          .select("PROVIDER_ID").distinct().withColumn("OWN_WRITER_3M", F.lit(1)))
target = (mv_rows.filter((F.col("month") >= F.lit(out_start)) & (F.col("month") <= F.lit(out_end)))
          .select("PROVIDER_ID").distinct().withColumn("TARGET", F.lit(1)))
sig_base = universe.join(own_lb, "PROVIDER_ID", "left").join(target, "PROVIDER_ID", "left").fillna(0, ["OWN_WRITER_3M", "TARGET"]).toPandas()
sd = fp.merge(sig_base, on="PROVIDER_ID", how="left")
sd["HCP_EXPERIENCE"] = np.where(sd._GRAD_YEAR <= row.anchor_year, row.anchor_year - sd._GRAD_YEAR, np.nan)   # N6 for c17
base_rate, nw = sd.TARGET.mean(), sd[sd.OWN_WRITER_3M == 0]
record("P1 KER-10 outcome built", len(sd) == n_hcp and nw.TARGET.mean() < base_rate,
       f"{SIG_COHORT}: look-back {lb_start}..{lb_end}, outcome {out_start}..{out_end}; writers {int(sd.TARGET.sum()):,} ({base_rate:.2%}); "
       f"new-writer view {len(nw):,} HCPs ({nw.TARGET.mean():.2%})")

In [0]:
NUMERIC = ["HCP_EXPERIENCE", "AFFL_DISTINCT_ACCT_COUNT", "ACCT_MD_DO_COUNT"]
# 07_2 direction (+1 / -1) where 07_2 shows a clear full-universe gap; None = not tested
# (CARD > 0 is only 10.3% vs 9.8% overall; the PRFL code flags and EMPLOYMENT_FLAG were compared among MDs or across 3 groups)
DIR_07_2 = {"HCP_DEGREE_NP_PA": -1, "HCP_GENDER_F": -1, "HCP_SECONDARY_SPEC_CARDIORENAL": 1, "HCP_EXPERIENCE": 1,
            "AFFL_NEPH_ACCT_COUNT": 1, "AFFL_CARD_ACCT_COUNT": None, "AFFL_DISTINCT_ACCT_COUNT": 1, "DIALYSIS_CENTER_FLAG": 1,
            "AFFL_ENDO_ACCT_COUNT": 1, "TEACHING_HOSP_FLAG": 1, "LONG_TERM_CARE_FACIL_FLAG": 1,
            "HCP_AFFL_DESCRIPTION_ORGANIZATION_HEALTH_SYSTEM": 1, "HCP_AFFL_DESCRIPTION_ORGANIZATION_ADMIN_ONLY": 1,
            "HCP_AFFL_DESCRIPTION_ORGANIZATION_ACCOUNTABLE_CARE_ORGANIZATIONS_ACOS": -1,
            "HCP_EMPLOYMENT_SELF_EMPLOYED_SOLO_PRACTICE": None, "HCP_MPA_INACTIVE": None, "HCP_PRACTICE_TYPE_RESIDENT": None,
            "HCP_PRACTICE_TYPE_NOT_CLASSIFIED": 1, "BEST_EMPLOYMENT_FLAG_Y": None, "BEST_EMPLOYMENT_FLAG_N": None,
            "BEST_EMPLOYMENT_FLAG_NA": None, "ACCT_MD_DO_COUNT": -1}

def spearman(x, y):
    return float(pd.Series(x).rank().corr(pd.Series(y).rank()))

rows = []
for c in FEATURE_COLS:
    s = sd[[c, "TARGET", "OWN_WRITER_3M"]].dropna(subset=[c])
    s_nw = s[s.OWN_WRITER_3M == 0]
    if c in NUMERIC:
        q = pd.qcut(s[c].rank(method="first"), 5, labels=False)
        qn = pd.qcut(s_nw[c].rank(method="first"), 5, labels=False)
        a, b = s[q == 4].TARGET.mean(), s[q == 0].TARGET.mean()
        na, nb = s_nw[qn == 4].TARGET.mean(), s_nw[qn == 0].TARGET.mean()
        kind, n_pos = "Q5 vs Q1", len(s)
    else:
        yes = s[c] > 0
        a, b = s[yes].TARGET.mean(), s[~yes].TARGET.mean()
        na, nb = s_nw[s_nw[c] > 0].TARGET.mean(), s_nw[~(s_nw[c] > 0)].TARGET.mean()
        kind, n_pos = "> 0 vs 0", int(yes.sum())
    ref = DIR_07_2[c]
    rows.append({"feature": c, "status": "Build" if c in BUILD_COLS else "Test", "fill_%": round(100 * len(s) / n_hcp, 1),
                 "HCPs_yes_or_filled": n_pos, "compare": kind, "rate_A_%": round(100 * a, 1), "rate_B_%": round(100 * b, 1),
                 "new_writer_A_%": round(100 * na, 1), "new_writer_B_%": round(100 * nb, 1),
                 "rho_KER10": round(spearman(s[c], s.TARGET), 3),
                 "direction_07_2": ref, "direction_holds": None if ref is None else bool(np.sign(a - b) == ref)})
sig = pd.DataFrame(rows)
tested = sig[sig.direction_holds.notna()]
flips = tested[tested.direction_holds == False].feature.tolist()
record("P2 signal direction holds on KER-10", not flips,
       f"{len(tested)} of {len(sig)} features tested; direction differs from 07_2 for: {flips if flips else 'none'}")
display(sig)

## Phase 3: Save one row per HCP x cohort and check the schema on two runs

In [0]:
def expand(feat_df, cohorts_df):
    """N10: repeat the HCP features for every cohort; N6: HCP_EXPERIENCE anchored to the cohort prediction year."""
    x = (feat_df.crossJoin(F.broadcast(cohorts_df.select("cohort_id", F.col("lookback_end_date").alias("SVC_DT_M"), "anchor_year")))
         .withColumn("HCP_EXPERIENCE", F.when(F.col("_GRAD_YEAR") <= F.col("anchor_year"), F.col("anchor_year") - F.col("_GRAD_YEAR")))
         .withColumn("HCP_COHORT_ID", F.concat_ws("_", "PROVIDER_ID", "cohort_id"))
         .withColumn("PULL_DT", F.current_date()))
    return x

OUT_COLS = KEYS + FEATURE_COLS + ["PULL_DT"]
out = expand(feat_hcp, coh).select(*OUT_COLS)
out.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(OUTPUT_PATH)
run1 = spark.read.format("delta").load(OUTPUT_PATH)
n_out = run1.count()
record("Output written", n_out == n_hcp * len(cohorts), f"{n_out:,} rows (= {n_hcp:,} x {len(cohorts)}) -> {OUTPUT_PATH}")
n_keys = run1.select("HCP_COHORT_ID").distinct().count()
record("S6 one row per HCP x cohort", n_keys == n_out, f"distinct HCP_COHORT_ID {n_keys:,}")

last2 = cohorts.cohort_id.iloc[-2:].tolist()
run2 = expand(feat_hcp, coh.filter(F.col("cohort_id").isin(last2))).select(*OUT_COLS)
same_schema = [(f.name, f.dataType.simpleString()) for f in run1.schema] == [(f.name, f.dataType.simpleString()) for f in run2.schema]
diff = run1.filter(F.col("cohort_id").isin(last2)).drop("PULL_DT").exceptAll(run2.drop("PULL_DT")).count()
record("S6 fixed schema on two runs", same_schema and diff == 0 and run1.columns == OUT_COLS,
       f"{len(OUT_COLS)} columns ({len(FEATURE_COLS)} features), schema identical: {same_schema}, differing rows on rerun of {last2}: {diff}")

# N6 checks: static features identical across cohorts; experience never negative and never uses a future grad year
hash_col = F.xxhash64(*[F.coalesce(F.col(c).cast("string"), F.lit("<null>")) for c in STATIC_COLS])
n_var = run1.groupBy("PROVIDER_ID").agg(F.countDistinct(hash_col).alias("h")).filter("h > 1").count()
n_neg = run1.filter("HCP_EXPERIENCE < 0").count()
record("S6 static values identical across cohorts; experience >= 0", n_var == 0 and n_neg == 0,
       f"HCPs whose static values differ between cohorts: {n_var}; rows with negative experience: {n_neg}")

In [0]:
def ok(*ids):
    return all(CHECKS[i]["passed"] for i in ids)
goals = [
    ("Sources read; MDM codes match the flag names (N1, N7)", ok("S1 MDM codes match flag names (N7)", "S1 one rank-1 BEST_AFFL row per BH_ID", "S1 coverage = 07_2")),
    ("Features reproduce 07_2 counts and the 3.1 draft values", ok("S2 one row per HCP", "S2 counts = 07_2 (within 0.5%)", "S2 same values as the 3.1 draft (within 0.5% of HCPs)")),
    ("Missing states (N8)", ok("S2 missing-state rules (N8)")),
    ("Signal direction from 07_2 holds on KER-10 writers (N12)", ok("P1 KER-10 outcome built", "P2 signal direction holds on KER-10")),
    ("One row per HCP x cohort, fixed schema, experience anchored (N6, N10)", ok("Output written", "S6 one row per HCP x cohort", "S6 fixed schema on two runs", "S6 static values identical across cohorts; experience >= 0")),
]
goal_df = pd.DataFrame(goals, columns=["goal", "achieved"])
check_df = pd.DataFrame([{"check": k, **v} for k, v in CHECKS.items()])
print(f"{int(check_df.passed.sum())} of {len(check_df)} checks passed")
display(check_df); display(goal_df)

In [0]:
final = spark.read.format("delta").load(OUTPUT_PATH)
last_c = cohorts.cohort_id.iloc[-1]
fr = final.filter(F.col("cohort_id") == last_c).select(
    *[F.avg(F.col(c).isNotNull().cast("double")).alias(f"{c}__filled") for c in FEATURE_COLS],
    *[F.avg((F.col(c) != 0).cast("double")).alias(f"{c}__nonzero") for c in FEATURE_COLS]).toPandas().T[0]
fill = pd.DataFrame({"feature": FEATURE_COLS, "status": ["Build" if c in BUILD_COLS else "Test" for c in FEATURE_COLS],
                     "filled_%": [round(100 * fr[f"{c}__filled"], 2) for c in FEATURE_COLS],
                     "non_zero_%": [round(100 * fr[f"{c}__nonzero"], 2) for c in FEATURE_COLS]})
print(f"{final.count():,} rows x {len(final.columns)} columns | fill rates for cohort {last_c} ({n_hcp:,} HCPs)")
display(fill)